In [0]:
 %sql
use catalog useastws

In [0]:
%sql
create volume if not exists default.deltavolume

In [0]:
%fs
mkdirs /Volumes/useastws/default/deltavolume/ordersdata1

res5: Boolean = true

In [0]:
from decimal import Decimal
from pyspark.sql.types import *
schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer_name", StringType()),
    StructField("product", StringType()),
    StructField("quantity", IntegerType()),
    StructField("unit_price", FloatType()),
    StructField("order_date", StringType()),
    StructField("status", StringType())
])

In [0]:
data = [
       (1001, 'Alice Johnson',  'Laptop',         1, 1299.99, '2024-01-15', 'Delivered'),
    (1002, 'Bob Smith',     'Wireless Mouse',  3,   29.99, '2024-01-18', 'Delivered'),
    (1003, 'Carol White',   'Monitor',         2,  399.99, '2024-02-02', 'Shipped'),
    (1004, 'David Brown',   'Keyboard',        1,   89.99, '2024-02-10', 'Processing'),
    (1005, 'Eva Martinez',  'Headphones',      2,  149.99, '2024-02-14', 'Delivered'),
    (1006, 'Frank Lee',     'Webcam',          1,   79.99, '2024-03-01', 'Cancelled'),
    (1007, 'Grace Kim',     'USB-C Hub',       4,   49.99, '2024-03-05', 'Shipped'),
    (1008, 'Henry Wilson',  'SSD Drive',       2,  129.99, '2024-03-12', 'Processing')
]

In [0]:
df = spark.createDataFrame(data, schema=schema)

In [0]:
df.show(3)

+--------+-------------+--------------+--------+----------+----------+---------+
|order_id|customer_name|       product|quantity|unit_price|order_date|   status|
+--------+-------------+--------------+--------+----------+----------+---------+
|    1001|Alice Johnson|        Laptop|       1|   1299.99|2024-01-15|Delivered|
|    1002|    Bob Smith|Wireless Mouse|       3|     29.99|2024-01-18|Delivered|
|    1003|  Carol White|       Monitor|       2|    399.99|2024-02-02|  Shipped|
+--------+-------------+--------------+--------+----------+----------+---------+
only showing top 3 rows


In [0]:
volpath = '/Volumes/useastws/default/deltavolume/ordersdata1/'

In [0]:
df.write.format('delta').mode('overwrite').save(volpath)

In [0]:
%sql
describe history delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-08-26T15:07:48Z,147836707444603,anooptu@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(474475763794151),0826-063441-ydade36q,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> 8, numOutputBytes -> 2825)",null,Databricks-Runtime/16.4.x-scala2.13


In [0]:
%sql
select * from delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

order_id,customer_name,product,quantity,unit_price,order_date,status
1001,Alice Johnson,Laptop,1,1299.99,2024-01-15,Delivered
1002,Bob Smith,Wireless Mouse,3,29.99,2024-01-18,Delivered
1003,Carol White,Monitor,2,399.99,2024-02-02,Shipped
1004,David Brown,Keyboard,1,89.99,2024-02-10,Processing
1005,Eva Martinez,Headphones,2,149.99,2024-02-14,Delivered
1006,Frank Lee,Webcam,1,79.99,2024-03-01,Cancelled
1007,Grace Kim,USB-C Hub,4,49.99,2024-03-05,Shipped
1008,Henry Wilson,SSD Drive,2,129.99,2024-03-12,Processing


In [0]:
from decimal import Decimal
from datetime import date

# Mixed data: order_ids 1003, 1005, 1007 already exist in the delta table (same keys)
# order_ids 1009, 1010, 1011 are brand new records
new_data = [
    # Existing keys (will update on merge)
    (1003, 'Carol White',   'Monitor',          3,  389.99, date(2024, 2, 2),  'Delivered'),
    (1005, 'Eva Martinez',  'Headphones',        1,  159.99, date(2024, 2, 14), 'Shipped'),
    (1007, 'Grace Kim',     'USB-C Hub',         6,  44.99,  date(2024, 3, 5),  'Delivered'),
    # New keys (will insert on merge)
    (1009, 'Isla Turner',   'Standing Desk',     1,  549.99, date(2024, 4, 1), 'Processing'),
    (1010, 'Jack Harris',   'Mechanical Keyboard',2, 119.99, date(2024, 4, 5),  'Shipped'),
    (1011, 'Karen Scott',   'Monitor Arm',       1,  69.99,  date(2024, 4, 8),  'Processing'),
]

df_new = spark.createDataFrame(new_data, schema=schema)
df_new.createOrReplaceTempView('orders_upsert')

df_new.show(truncate=False)
print(f"Total rows in view: {df_new.count()} (3 existing keys + 3 new keys)")

+--------+-------------+-------------------+--------+----------+----------+----------+
|order_id|customer_name|product            |quantity|unit_price|order_date|status    |
+--------+-------------+-------------------+--------+----------+----------+----------+
|1003    |Carol White  |Monitor            |3       |389.99    |2024-02-02|Delivered |
|1005    |Eva Martinez |Headphones         |1       |159.99    |2024-02-14|Shipped   |
|1007    |Grace Kim    |USB-C Hub          |6       |44.99     |2024-03-05|Delivered |
|1009    |Isla Turner  |Standing Desk      |1       |549.99    |2024-04-01|Processing|
|1010    |Jack Harris  |Mechanical Keyboard|2       |119.99    |2024-04-05|Shipped   |
|1011    |Karen Scott  |Monitor Arm        |1       |69.99     |2024-04-08|Processing|
+--------+-------------+-------------------+--------+----------+----------+----------+

Total rows in view: 6 (3 existing keys + 3 new keys)


MERGE SOURCE INTO TARGET. UPDATE IF KEY MATCH, ELSE INSERT

In [0]:
%sql
merge into delta.`/Volumes/useastws/default/deltavolume/ordersdata1/` as target
using orders_upsert as source
on target.order_id = source.order_id
when matched then update set *
when not matched then insert * ;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
6,3,0,3


In [0]:
%sql
describe history delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-08-26T15:08:11Z,147836707444603,anooptu@gmail.com,MERGE,"Map(predicate -> [""(order_id#1234 = order_id#1005)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(474475763794151),0826-063441-ydade36q,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 9865, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 3, executionTimeMs -> 6591, materializeSourceTimeMs -> 235, numTargetRowsInserted -> 3, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3512, numTargetRowsUpdated -> 3, numOutputRows -> 6, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 6, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2742)",null,Databricks-Runtime/16.4.x-scala2.13
0,2026-08-26T15:07:48Z,147836707444603,anooptu@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(474475763794151),0826-063441-ydade36q,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> 8, numOutputBytes -> 2825)",null,Databricks-Runtime/16.4.x-scala2.13


In [0]:
%sql
select * from delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

order_id,customer_name,product,quantity,unit_price,order_date,status
1001,Alice Johnson,Laptop,1,1299.99,2024-01-15,Delivered
1002,Bob Smith,Wireless Mouse,3,29.99,2024-01-18,Delivered
1004,David Brown,Keyboard,1,89.99,2024-02-10,Processing
1006,Frank Lee,Webcam,1,79.99,2024-03-01,Cancelled
1008,Henry Wilson,SSD Drive,2,129.99,2024-03-12,Processing
1009,Isla Turner,Standing Desk,1,549.99,2024-04-01,Processing
1010,Jack Harris,Mechanical Keyboard,2,119.99,2024-04-05,Shipped
1011,Karen Scott,Monitor Arm,1,69.99,2024-04-08,Processing
1003,Carol White,Monitor,3,389.99,2024-02-02,Delivered
1005,Eva Martinez,Headphones,1,159.99,2024-02-14,Shipped


In [0]:
from decimal import Decimal
from datetime import date

# Mixed data: order_ids 1003, 1005, 1007 already exist in the delta table (same keys)
# order_ids 1009, 1010, 1011 are brand new records
new_data = [
    # Existing keys (will update on merge)
    (1003, 'Carol White',   'Monitor',          3,  389.99, date(2024, 2, 2),  'Delivered'),
    (1005, 'Eva Martinez',  'Headphones',        1,  159.99, date(2024, 2, 14), 'Shipped'),
    (1007, 'Grace Kim',     'USB-C Hub',         6,  44.99,  date(2024, 3, 5),  'Delivered'),
    # New keys (will insert on merge)
    (1009, 'Isla Turner',   'Standing Desk',     1,  549.99, date(2024, 4, 1), 'Processing'),
    (1010, 'Jack Harris',   'Mechanical Keyboard',2, 119.99, date(2024, 4, 5),  'Shipped'),
    (1011, 'Karen Scott',   'Monitor Arm',       1,  69.99,  date(2024, 4, 8),  'Processing'),
]

df_new = spark.createDataFrame(new_data, schema=schema)
df_new.createOrReplaceTempView('orders_upsert')

df_new.show(truncate=False)
print(f"Total rows in view: {df_new.count()} (3 existing keys + 3 new keys)")

+--------+-------------+-------------------+--------+----------+----------+----------+
|order_id|customer_name|product            |quantity|unit_price|order_date|status    |
+--------+-------------+-------------------+--------+----------+----------+----------+
|1003    |Carol White  |Monitor            |3       |389.99    |2024-02-02|Delivered |
|1005    |Eva Martinez |Headphones         |1       |159.99    |2024-02-14|Shipped   |
|1007    |Grace Kim    |USB-C Hub          |6       |44.99     |2024-03-05|Delivered |
|1009    |Isla Turner  |Standing Desk      |1       |549.99    |2024-04-01|Processing|
|1010    |Jack Harris  |Mechanical Keyboard|2       |119.99    |2024-04-05|Shipped   |
|1011    |Karen Scott  |Monitor Arm        |1       |69.99     |2024-04-08|Processing|
+--------+-------------+-------------------+--------+----------+----------+----------+

Total rows in view: 6 (3 existing keys + 3 new keys)


MERGE WITH DELETE IF EXISTING AND CONDITION MET, ELSE UPDATE OR INSERT AS EARLIER

In [0]:
%sql
create or replace temp view incoming_flag as    
select * from values
(1003, 'Carol White',   'Monitor',    3,  389.99, ('2024-02-02'),  'Delivered',FALSE),
(1005, 'Eva Martinez',  'Headphones',  1,  159.99, ('2024-02-14'), 'Shipped',TRUE)
as t(order_id , customer_name,product, quantity, unit_price,order_date, status,is_deleted)



In [0]:
%sql
select * from incoming_flag

order_id,customer_name,product,quantity,unit_price,order_date,status,is_deleted
1003,Carol White,Monitor,3,389.99,2024-02-02,Delivered,false
1005,Eva Martinez,Headphones,1,159.99,2024-02-14,Shipped,true


In [0]:
%sql
merge into delta.`/Volumes/useastws/default/deltavolume/ordersdata1/` as target
using incoming_flag as source
on target.order_id = source.order_id
when matched and source.is_deleted = TRUE   then delete 
when matched  then update set *  
when not matched then insert  * ;
     

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,1,0


In [0]:
%sql
select * from delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

order_id,customer_name,product,quantity,unit_price,order_date,status
1001,Alice Johnson,Laptop,1,1299.99,2024-01-15,Delivered
1002,Bob Smith,Wireless Mouse,3,29.99,2024-01-18,Delivered
1004,David Brown,Keyboard,1,89.99,2024-02-10,Processing
1006,Frank Lee,Webcam,1,79.99,2024-03-01,Cancelled
1008,Henry Wilson,SSD Drive,2,129.99,2024-03-12,Processing
1007,Grace Kim,USB-C Hub,6,44.99,2024-03-05,Delivered
1009,Isla Turner,Standing Desk,1,549.99,2024-04-01,Processing
1010,Jack Harris,Mechanical Keyboard,2,119.99,2024-04-05,Shipped
1011,Karen Scott,Monitor Arm,1,69.99,2024-04-08,Processing
1003,Carol White,Monitor,3,389.99,2024-02-02,Delivered


In [0]:
%sql
create or replace temp view incoming_snapshot as    
select * from values
(1003, 'Carol White',   'Monitor',    3,  389.99, ('2024-02-02'),  'Delivered'),
(1012, 'Eva Martinez',  'Headphones',  1,  159.99, ('2024-02-14'), 'Shipped')
as t(order_id , customer_name,product, quantity, unit_price,order_date, status);

MERGE SUCH THAT, ONLY THE SOURCE ROWS remains in the target. 

In [0]:
%sql
merge into delta.`/Volumes/useastws/default/deltavolume/ordersdata1/` as target
using incoming_snapshot as source
on target.order_id = source.order_id
when matched then update set *
when not matched then insert * 
when not matched by source then delete;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
11,1,9,1


In [0]:
%sql
describe history delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-08-26T15:08:41Z,147836707444603,anooptu@gmail.com,MERGE,"Map(predicate -> [""(order_id#5109 = order_id#5095)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [{""actionType"":""delete""}], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(474475763794151),0826-063441-ydade36q,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 9, numTargetFilesAdded -> 2, numTargetBytesAdded -> 4920, numTargetBytesRemoved -> 5405, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 3534, materializeSourceTimeMs -> 158, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1327, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 1, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 2, numTargetRowsNotMatchedBySourceDeleted -> 9, rewriteTimeMs -> 1956)",null,Databricks-Runtime/16.4.x-scala2.13
3,2026-08-26T15:08:29Z,147836707444603,anooptu@gmail.com,MERGE,"Map(predicate -> [""(order_id#3586 = order_id#3571)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""is_deleted#3578: boolean"",""actionType"":""delete""},{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(474475763794151),0826-063441-ydade36q,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 1, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2453, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 4395, materializeSourceTimeMs -> 129, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 1, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1755, numTargetRowsUpdated -> 1, numOutputRows -> 1, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2465)",null,Databricks-Runtime/16.4.x-scala2.13
2,2026-08-26T15:08:20Z,147836707444603,anooptu@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(474475763794151),0826-063441-ydade36q,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 12690, p25FileSize -> 2952, numDeletionVectorsRemoved -> 1, minFileSize -> 2952, numAddedFiles -> 1, maxFileSize -> 2952, p75FileSize -> 2952, p50FileSize -> 2952, numAddedBytes -> 2952)",null,Databricks-Runtime/16.4.x-scala2.13
1,2026-08-26T15:08:11Z,147836707444603,anooptu@gmail.com,MERGE,"Map(predicate -> [""(order_id#1234 = order_id#1005)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(474475763794151),0826-063441-ydade36q,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 9865, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 3, executionTimeMs -> 6591, materializeSourceTimeMs -> 235, numTargetRowsInserted -> 3, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3512, numTargetRowsUpdated -> 3, numOutputRows -> 6, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 6, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2742)",null,Databricks-R

In [0]:
%sql
select * from delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

order_id,customer_name,product,quantity,unit_price,order_date,status
1012,Eva Martinez,Headphones,1,159.99,2024-02-14,Shipped
1003,Carol White,Monitor,3,389.99,2024-02-02,Delivered


In [0]:
%sql
select * from delta.`/Volumes/useastws/default/deltavolume/ordersdata1/` version as of 2

order_id,customer_name,product,quantity,unit_price,order_date,status
1001,Alice Johnson,Laptop,1,1299.99,2024-01-15,Delivered
1002,Bob Smith,Wireless Mouse,3,29.99,2024-01-18,Delivered
1004,David Brown,Keyboard,1,89.99,2024-02-10,Processing
1006,Frank Lee,Webcam,1,79.99,2024-03-01,Cancelled
1008,Henry Wilson,SSD Drive,2,129.99,2024-03-12,Processing
1003,Carol White,Monitor,3,389.99,2024-02-02,Delivered
1005,Eva Martinez,Headphones,1,159.99,2024-02-14,Shipped
1007,Grace Kim,USB-C Hub,6,44.99,2024-03-05,Delivered
1009,Isla Turner,Standing Desk,1,549.99,2024-04-01,Processing
1010,Jack Harris,Mechanical Keyboard,2,119.99,2024-04-05,Shipped


Dataframe format merge

In [0]:
snapshot = [
    (1003, 'Carol White',   'Monitor',    3,  389.99, ('2024-02-02'),  'Delivered'),
(1014, 'Eva Martinez1',  'Headphones',  2,  159.99, ('2024-02-14'), 'Shipped')
]
snapshot_df = spark.createDataFrame(snapshot,schema=schema)

In [0]:
from delta.tables import DeltaTable
path = '/Volumes/useastws/default/deltavolume/ordersdata1/'
delta_table = DeltaTable.forPath(spark, path)

In [0]:
(delta_table.alias('t')
 .merge(snapshot_df.alias('s'),'t.order_id = s.order_id')
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .whenNotMatchedBySourceDelete()
        .execute()
 )

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
describe history delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-08-26T15:25:59Z,147836707444603,anooptu@gmail.com,MERGE,"Map(predicate -> [""(order_id#7489 = order_id#7186)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [{""actionType"":""delete""}], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(474475763794151),0826-063441-ydade36q,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 1, numTargetFilesAdded -> 2, numTargetBytesAdded -> 4927, numTargetBytesRemoved -> 4920, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 2811, materializeSourceTimeMs -> 96, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 964, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 2, numTargetRowsNotMatchedBySourceDeleted -> 1, rewriteTimeMs -> 1707)",null,Databricks-Runtime/16.4.x-scala2.13
4,2026-08-26T15:08:41Z,147836707444603,anooptu@gmail.com,MERGE,"Map(predicate -> [""(order_id#5109 = order_id#5095)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [{""actionType"":""delete""}], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(474475763794151),0826-063441-ydade36q,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 9, numTargetFilesAdded -> 2, numTargetBytesAdded -> 4920, numTargetBytesRemoved -> 5405, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 3534, materializeSourceTimeMs -> 158, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1327, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 1, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 2, numTargetRowsNotMatchedBySourceDeleted -> 9, rewriteTimeMs -> 1956)",null,Databricks-Runtime/16.4.x-scala2.13
3,2026-08-26T15:08:29Z,147836707444603,anooptu@gmail.com,MERGE,"Map(predicate -> [""(order_id#3586 = order_id#3571)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""is_deleted#3578: boolean"",""actionType"":""delete""},{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(474475763794151),0826-063441-ydade36q,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 1, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2453, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 4395, materializeSourceTimeMs -> 129, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 1, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1755, numTargetRowsUpdated -> 1, numOutputRows -> 1, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2465)",null,Databricks-Runtime/16.4.x-scala2.13
2,2026-08-26T15:08:20Z,147836707444603,anooptu@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(474475763794151),0826-063441-ydade36q,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 12690, p25FileSize -> 2952, numDeletionVectorsRemoved -> 1, minFileSize -> 2952, numAddedFiles -> 1, maxFileSize -> 2952, p75FileSize -> 2952, p50FileSize -> 2952, numAddedBytes

In [0]:
delta_table.toDF().display()

order_id,customer_name,product,quantity,unit_price,order_date,status
1014,Eva Martinez1,Headphones,2,159.99,2024-02-14,Shipped
1003,Carol White,Monitor,3,389.99,2024-02-02,Delivered


In [0]:
display(spark.sql(f'select * from delta.`{path}`'))

order_id,customer_name,product,quantity,unit_price,order_date,status
1014,Eva Martinez1,Headphones,2,159.99,2024-02-14,Shipped
1003,Carol White,Monitor,3,389.99,2024-02-02,Delivered


In [0]:
display(spark.sql(f'describe history delta.`{path}`'))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-08-26T15:25:59Z,147836707444603,anooptu@gmail.com,MERGE,"Map(predicate -> [""(order_id#7489 = order_id#7186)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [{""actionType"":""delete""}], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(474475763794151),0826-063441-ydade36q,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 1, numTargetFilesAdded -> 2, numTargetBytesAdded -> 4927, numTargetBytesRemoved -> 4920, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 2811, materializeSourceTimeMs -> 96, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 964, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 2, numTargetRowsNotMatchedBySourceDeleted -> 1, rewriteTimeMs -> 1707)",null,Databricks-Runtime/16.4.x-scala2.13
4,2026-08-26T15:08:41Z,147836707444603,anooptu@gmail.com,MERGE,"Map(predicate -> [""(order_id#5109 = order_id#5095)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [{""actionType"":""delete""}], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(474475763794151),0826-063441-ydade36q,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 9, numTargetFilesAdded -> 2, numTargetBytesAdded -> 4920, numTargetBytesRemoved -> 5405, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 3534, materializeSourceTimeMs -> 158, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1327, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 1, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 2, numTargetRowsNotMatchedBySourceDeleted -> 9, rewriteTimeMs -> 1956)",null,Databricks-Runtime/16.4.x-scala2.13
3,2026-08-26T15:08:29Z,147836707444603,anooptu@gmail.com,MERGE,"Map(predicate -> [""(order_id#3586 = order_id#3571)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""is_deleted#3578: boolean"",""actionType"":""delete""},{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(474475763794151),0826-063441-ydade36q,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 1, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2453, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 4395, materializeSourceTimeMs -> 129, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 1, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1755, numTargetRowsUpdated -> 1, numOutputRows -> 1, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2465)",null,Databricks-Runtime/16.4.x-scala2.13
2,2026-08-26T15:08:20Z,147836707444603,anooptu@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(474475763794151),0826-063441-ydade36q,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 12690, p25FileSize -> 2952, numDeletionVectorsRemoved -> 1, minFileSize -> 2952, numAddedFiles -> 1, maxFileSize -> 2952, p75FileSize -> 2952, p50FileSize -> 2952, numAddedBytes